In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from datetime import datetime
import os

In [ ]:
# Import the CSV file for trip data
df = pd.read_excel('Provided DB_Final.xlsx', sheet_name = 'Each_ADB_details')

# Convert recordTime column to datetime format
df['recordTime'] = pd.to_datetime(df['recordTime'], format='%Y%m%d%H%M%S')
# Convert datetime format to Unix timestamps
df['unixTime'] = (df['recordTime'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')
df['unixStart'] = (df['move_start_time'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')

In [ ]:
def get_interval_before_adb(df_both, ADB_Times, interval):
    # Convert interval to timedelta object
    interval_td = pd.Timedelta(interval)
    
    # Convert end_unix column to datetime64[ns]
    df_both['end_datetime'] = pd.to_datetime(df_both['end_unix'], unit='s')
    
    # Create a mask to select rows in df_both that have an end_unix value in ADB_Times
    mask = df_both['end_unix'].isin(ADB_Times)
    ADB_data = df_both[mask]
    
    interval_dfs = []
    means = []
    for idx, row in ADB_data.iterrows():
        # Get the start and end times of the interval before the ADB event
        end_time = row['end_datetime']
        start_time = end_time - interval_td
        
        # Select rows in df_both that fall within the interval before the ADB event
        mask = (df_both['end_datetime'] >= start_time) & (df_both['end_datetime'] <= end_time)
        interval_data = df_both.loc[mask]
        
        # Append interval_data to interval_dfs
        interval_dfs.append(interval_data)
        mean_dict = interval_data.mean().to_dict()
        means.append(mean_dict)
      
    interval_df = pd.concat(interval_dfs, axis=0)
    means_before = pd.DataFrame(means) 
    
    return interval_df, means_before

In [ ]:
def get_interval_after_adb(df_both, ADB_Times, interval):
    # Convert interval to timedelta object
    interval_td = pd.Timedelta(interval)
    
    # Convert end_unix column to datetime64[ns]
    df_both['end_datetime'] = pd.to_datetime(df_both['end_unix'], unit='s')
    
    # Create a mask to select rows in df_both that have an end_unix value in ADB_Times
    mask = df_both['end_unix'].isin(ADB_Times)
    ADB_data = df_both[mask]
    
    interval_dfs = []
    
    for idx, row in ADB_data.iterrows():
        # Get the start and end times of the interval after the ADB event
        start_time = row['end_datetime']
        end_time = start_time + interval_td
        
        # Select rows in df_both that fall within the interval after the ADB event
        mask = (df_both['end_datetime'] >= start_time) & (df_both['end_datetime'] < end_time)
        interval_data = df_both.loc[mask]
        
        # Append interval_data to interval_dfs
        interval_dfs.append(interval_data)
    
    # Concatenate the individual DataFrames in interval_dfs into a single DataFrame
    interval_df = pd.concat(interval_dfs, axis=0)
    
    return interval_df

In [ ]:
####################
# BASELINE before recording #
####################
def get_baseline_hrv(df_hrv, ADB_Starts, interval):
    interval_td = pd.Timedelta(interval)
    df_hrv['end_datetime'] = pd.to_datetime(df_hrv['end_unix'], unit='s')
    
    intervals = []
    means = []
    for start_time in ADB_Starts:
        ADB_start = df_hrv[df_hrv['end_unix']==start_time]
        try:
            end_time = ADB_start['end_datetime'].iloc[0]
            start_time = ADB_start['end_datetime'].iloc[0] - interval_td
        except IndexError:
            print(interval)
            continue
        # Select rows in df_hrv that fall within the interval
        interval_df = df_hrv[(df_hrv['end_datetime'] >= start_time) & (df_hrv['end_datetime'] <= end_time)]
        intervals.append(interval_df)
        mean_dict = interval_df.mean().to_dict()
        means.append(mean_dict)
    baseline_df = pd.concat(intervals)
    means_baseline = pd.DataFrame(means)
    return baseline_df, means_baseline

In [ ]:
# filename = f"Driver_{1}.csv"
# df_hrv = pd.read_csv(filename)
# ADB_Starts = ADB_dict[1]['unixStart'].tolist()
# baseline, means_baseline = get_baseline_hrv(df_hrv, ADB_Starts, '5 minutes')
# baseline_ADBs.append(baseline)
# baseline_means.append(means_baseline)

In [ ]:
df_ADB = pd.DataFrame({'NO': df.iloc[:,0].astype(int), 'unixTime': df['unixTime'], 'unixStart': df['unixStart']})
driver_groups = df_ADB.groupby('NO')
df_ADB['unixStart'] = driver_groups['unixStart'].transform('first')
# Create ADB_dict with all drivers
ADB_dict = {driver: group[['unixTime', 'unixStart']] for driver, group in driver_groups}
print(ADB_dict)
# Remove the first row for the last driver
ADB_dict[31] = ADB_dict[31].iloc[1:]
ADB_dict[31]['unixStart'] = 1608200539
# print(ADB_dict)

In [ ]:
print(ADB_dict)

In [ ]:
before_ADBs = []
before_means = []
after_ADBs = []
baseline_ADBs = []
baseline_means = []

folder_path = "final_data" # path to the folder containing the CSV files

for i in ADB_dict:
    print(i)
    filename = os.path.join(folder_path, f"Final_{i}.csv") # include the path to the folder in the filename
    df_hrv = pd.read_csv(filename)
    ADB_Times = ADB_dict[i]['unixTime'].tolist()
    ADB_Starts = ADB_dict[i]['unixStart'].tolist()
    
    before, means_before = get_interval_before_adb(df_hrv, ADB_Times, '5 minutes')
    before_ADBs.append(before)
    before_means.append(means_before)
    
    after = get_interval_after_adb(df_hrv, ADB_Times, '5 minutes')
    after_ADBs.append(after)
    
    baseline, means_baseline = get_baseline_hrv(df_hrv, ADB_Starts, '5 minutes')
    baseline_ADBs.append(baseline)
    baseline_means.append(means_baseline)

    # Remove rows from before_means where the corresponding row is not present in baseline_means
    baseline_means = [df.loc[df.index.isin(before_means[i].index)] for i, df in enumerate(baseline_means)]

In [ ]:
folder_path = "final_data" # path to the folder containing the CSV files
table = []
for i in ADB_dict:
    print(i)
    filename = os.path.join(folder_path, f"Final_{i}.csv") # include the path to the folder in the filename
    df_hrv = pd.read_csv(filename)
    ADB_Times = ADB_dict[i]['unixTime'].tolist()
    ADB_Starts = ADB_dict[i]['unixStart'].tolist()
    trip_start = ADB_Starts[0]
    trip_end = ADB_Times[-1]
    ecg_start = df_hrv['start_time'][0]
    ecg_end = df_hrv['end_time'].iloc[-1]
    # Create a dictionary with the relevant data for the current driver
    data = {'driver': [i],
            'ecg_start': [ecg_start],
            'ecg_end': [ecg_end],
            'trip_start': [trip_start],
            'trip_end': [trip_end]}
    
    # Append the dictionary to the list of tables
    table.append(pd.DataFrame(data))

# Concatenate all tables into a single DataFrame
result = pd.concat(table)

# Print the resulting table
print(result)

In [ ]:
result.to_csv('start_end_data.csv', index=False)

In [ ]:
before_means_df = pd.concat(before_means, ignore_index=True)
before_means_df = before_means_df.iloc[:, 3:26]
baseline_means_df = pd.concat(baseline_means, ignore_index=True)
baseline_means_df = baseline_means_df.iloc[:,3:26]

before_df = pd.concat(before_ADBs)
after_df = pd.concat(after_ADBs)
baseline_df = pd.concat(baseline_ADBs)

ADB_before = before_df.iloc[:,5:28]
ADB_after = after_df.iloc[:,5:28]
ADB_baseline = baseline_df.iloc[:,5:28]

In [ ]:
from scipy.stats import shapiro

df_missing = before_means_df.dropna()
before_means_df = df_missing.fillna(df_missing.mean())
df_missing = baseline_means_df.dropna()
baseline_means_df = df_missing.fillna(df_missing.mean())

for column in before_means_df:
    stat, p = shapiro(before_means_df[column])
    print(f"{column}: stat={stat:.4f}, p={p:.4f}")
for column in baseline_means_df:
    stat, p = shapiro(baseline_means_df[column])
    print(f"{column}: stat={stat:.4f}, p={p:.4f}")

In [ ]:
from scipy.stats import wilcoxon
p_values = []
for column in baseline_means_df:
    stat, p = wilcoxon(baseline_means_df[column], before_means_df[column], zero_method = 'zsplit',alternative = 'two-sided', method = 'auto')
    p_values.append(p)
    print(f"{column}: stat={stat:.4f}, p={p:.4f}")

In [ ]:
before_mean = ADB_before.mean()
before_std = ADB_before.std()
before_both = pd.concat([before_mean, before_std], axis=1, keys=["mean", "std"])

# Create a new DataFrame with a MultiIndex
title = "Pre-event"
index = pd.MultiIndex.from_product([[title], before_both.columns])
before_both = pd.DataFrame(before_both.values, columns=index)

In [ ]:
baseline_mean = ADB_baseline.mean()
baseline_std = ADB_baseline.std()
baseline_both = pd.concat([baseline_mean, baseline_std], axis=1, keys=["mean", "std"])

# Create a new DataFrame with a MultiIndex
title = "Baseline"
index = pd.MultiIndex.from_product([[title], baseline_both.columns])
baseline_both = pd.DataFrame(baseline_both.values, columns=index)

In [ ]:
titles = ADB_before.columns.tolist()
titles_df = pd.DataFrame({'features': titles})
# Create a new DataFrame with a MultiIndex
title = "HRV"
index = pd.MultiIndex.from_product([[title], titles_df.columns])
titles_df = pd.DataFrame(titles_df.values, columns=index)

hrv_final = pd.concat([titles_df, baseline_both,before_both], axis=1)
hrv_final[('Wilcoxon', 'p-value')] = p_values
hrv_final = hrv_final.round(decimals=2)
hrv_final.to_csv("Final_compare.csv", index=False)
print(hrv_final)